In [11]:
%pip install requests pandas

Note: you may need to restart the kernel to use updated packages.


In [12]:
import requests
import pandas as pd

CHEMBL = "https://www.ebi.ac.uk/chembl/api/data"
PUBCHEM = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

In [13]:
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

CHEMBL = "https://www.ebi.ac.uk/chembl/api/data"
PUBCHEM = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

session = requests.Session()
retry = Retry(total=3, backoff_factor=2, status_forcelist=[500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retry))

def get_json(url, params=None):
    """Appelle une API et renvoie le JSON, ou None en cas d'erreur."""
    try:
        r = session.get(url, params=params, timeout=90)
        r.raise_for_status()
        return r.json()
    except requests.RequestException as e:
        print("Erreur API :", e)
        return None

In [14]:
def chercher_cible_chembl(nom_cible):
    print(f"=== Cible '{nom_cible}' dans ChEMBL ===")
    data = get_json(f"{CHEMBL}/target.json",
                    {"target_synonym__icontains": nom_cible,
                     "organism": "Homo sapiens",
                     "target_type": "SINGLE PROTEIN"})
    if not data or not data.get("targets"):
        print("Aucune cible trouvée.")
        return None
    t = data["targets"][0]
    print(f"Nom : {t['pref_name']} | ID : {t['target_chembl_id']} | Type : {t['target_type']}")
    return t["target_chembl_id"]

In [15]:
def chercher_molecule_chembl(nom_molecule):
    print(f"=== Molécule '{nom_molecule}' dans ChEMBL ===")
    data = get_json(f"{CHEMBL}/molecule.json", {"pref_name__iexact": nom_molecule})
    if not data or not data.get("molecules"):
        print("Molécule non trouvée.")
        return None
    m = data["molecules"][0]
    print(f"Nom : {m['pref_name']} | ID : {m['molecule_chembl_id']}")
    return m["molecule_chembl_id"]

In [16]:
def activites_cible_chembl(target_id, type_mesure="IC50", n=10):
    print(f"=== Activités {type_mesure} sur {target_id} (ChEMBL) ===")
    data = get_json(f"{CHEMBL}/activity.json",
                    {"target_chembl_id": target_id,
                     "standard_type": type_mesure,
                     "standard_relation": "=",
                     "standard_units": "nM",
                     "pchembl_value__isnull": "false",
                     "order_by": "-pchembl_value",
                     "limit": 100})
    if not data or not data.get("activities"):
        print("Aucune activité trouvée.")
        return None
    cols = ["molecule_chembl_id", "standard_type", "standard_value",
            "standard_units", "pchembl_value", "assay_chembl_id", "document_chembl_id"]
    df = pd.DataFrame(data["activities"])[cols]
    df = df.drop_duplicates(subset="molecule_chembl_id").head(n)
    return df

In [17]:
def chercher_molecule_pubchem(nom_molecule):
    print(f"=== Molécule '{nom_molecule}' dans PubChem ===")
    data = get_json(f"{PUBCHEM}/compound/name/{nom_molecule}/property/MolecularWeight,MolecularFormula,IUPACName/JSON")
    if not data:
        print("Molécule non trouvée.")
        return None
    props = data["PropertyTable"]["Properties"][0]
    print(f"Formule : {props['MolecularFormula']} | Poids : {props['MolecularWeight']} g/mol")
    print(f"Nom IUPAC : {props['IUPACName']}")
    return props

In [18]:
cible_id = chercher_cible_chembl("EGFR")
mol_id = chercher_molecule_chembl("gefitinib")
if cible_id:
    display(activites_cible_chembl(cible_id))
pubchem_info = chercher_molecule_pubchem("gefitinib")

=== Cible 'EGFR' dans ChEMBL ===
Nom : Epidermal growth factor receptor | ID : CHEMBL203 | Type : SINGLE PROTEIN
=== Molécule 'gefitinib' dans ChEMBL ===
Nom : GEFITINIB | ID : CHEMBL939
=== Activités IC50 sur CHEMBL203 (ChEMBL) ===


,molecule_chembl_id,standard_type,standard_value,standard_units,pchembl_value,assay_chembl_id,document_chembl_id
0,CHEMBL176582,IC50,0.01,nM,11.00,CHEMBL683040,CHEMBL1129035
1,CHEMBL4650319,IC50,0.01,nM,11.00,CHEMBL5260456,CHEMBL5257147
2,CHEMBL5270693,IC50,0.01,nM,11.00,CHEMBL6089387,CHEMBL6087440
3,CHEMBL5790648,IC50,0.016,nM,10.80,CHEMBL5736969,CHEMBL5728542
4,CHEMBL5746224,IC50,0.017,nM,10.77,CHEMBL5736969,CHEMBL5728542
6,CHEMBL5997498,IC50,0.02,nM,10.70,CHEMBL5736967,CHEMBL5728542
8,CHEMBL180022,IC50,0.02,nM,10.70,CHEMBL4670930,CHEMBL4665812
10,CHEMBL4537790,IC50,0.02,nM,10.70,CHEMBL5153037,CHEMBL5150044
11,CHEMBL3353410,IC50,0.02,nM,10.70,CHEMBL4670929,CHEMBL4665812
12,CHEMBL6054692,IC50,0.021,nM,10.68,CHEMBL5736969,CHEMBL5728542


=== Molécule 'gefitinib' dans PubChem ===
Formule : C22H24ClFN4O3 | Poids : 446.9 g/mol
Nom IUPAC : N-(3-chloro-4-fluorophenyl)-7-methoxy-6-(3-morpholin-4-ylpropoxy)quinazolin-4-amine
